# LC13 — Neural networks, and the discipline of evaluation (self-paced, ~50 min)

Two topics that belong together. First: the multilayer perceptron — the "hello world" of neural networks — applied to the load problem, including the unglamorous detail that decides whether it works at all (feature scaling). Second, and more important for your projects: the **evaluation discipline** that separates results from wishful thinking. Material appears in **Quiz 4**; this is the last taught content before your project period.

In [ ]:
%pip install scikit-learn pandas pyarrow matplotlib --quiet
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
df = pd.read_parquet("../data/svedala-year/svedala_hourly.parquet")
y = df["ZON_MITT"].interpolate(limit=3)
X = pd.DataFrame(index=df.index)
X["lag24"], X["lag168"] = y.shift(24), y.shift(168)
X["temp"] = df["temp_mid"].interpolate(limit=3)
X["hour_sin"] = np.sin(2*np.pi*df.index.hour/24); X["hour_cos"] = np.cos(2*np.pi*df.index.hour/24)
X["dow"] = df.index.dayofweek; X["workday"] = (X["dow"] < 5).astype(int)
data = X.join(y.rename("target")).dropna()
train = data.loc[:"2025-08-31"]; val = data.loc["2025-09-01":"2025-09-30"]; test = data.loc["2025-10-01":]
Xtr, ytr = train.drop(columns="target"), train["target"]
Xva, yva = val.drop(columns="target"), val["target"]
Xte, yte = test.drop(columns="target"), test["target"]
print(f"train {len(Xtr)} | validation {len(Xva)} | test {len(Xte)} — three sets, three jobs")

## 1. The MLP — and where the scaler really earns its keep

A neural network is layers of weighted sums squeezed through nonlinearities, trained by gradient descent. Textbooks say "always scale your features" — and it costs nothing, so we do. But watch the honest comparison below: with modern defaults (relu + adam) the unscaled model survives; the two land within noise of each other. The always-scale wisdom comes from configurations where unscaled training collapses outright — and from the fact that you rarely know in advance which configuration you are in.

The deeper reason the `StandardScaler` sits inside a **pipeline** is not speed — it is *honesty*: the scaler is fitted on training data only and travels with the model, so the test set's statistics can never leak into preprocessing.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
raw = MLPRegressor(hidden_layer_sizes=(32,16), max_iter=300, random_state=0).fit(Xtr, ytr)
scaled = make_pipeline(StandardScaler(),
         MLPRegressor(hidden_layer_sizes=(32,16), max_iter=300, random_state=0)).fit(Xtr, ytr)
mae_raw = float(np.abs(raw.predict(Xva)-yva).mean())
mae_scaled = float(np.abs(scaled.predict(Xva)-yva).mean())
print(f"MLP unscaled  val MAE: {mae_raw:7.1f} MW")
print(f"MLP + scaler  val MAE: {mae_scaled:7.1f} MW")
print("-> close on this problem with these defaults; the pipeline is kept for the leakage guarantee")

Same architecture, dramatically different outcome — preprocessing IS part of the model. (This is also why the pipeline object exists: the scaler is fitted on training data only and travels with the model, so the test set never leaks into the preprocessing.)

## 2. Train / validation / test — three sets, three jobs

- **Train**: fit parameters.
- **Validation**: choose between models and settings. You may look as often as you like — and every look costs a little honesty, which is why the third set exists.
- **Test**: touched ONCE, at the end, to report the number. A test set consulted during development is just a second validation set wearing a costume.

Model selection happens on validation:

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
candidates = {
    "MLP (32,16)": scaled,
    "MLP (64,32)": make_pipeline(StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(64,32), max_iter=300, random_state=0)).fit(Xtr, ytr),
    "gbdt": HistGradientBoostingRegressor(random_state=0).fit(Xtr, ytr),
}
val_mae = {k: float(np.abs(m.predict(Xva)-yva).mean()) for k, m in candidates.items()}
print(pd.Series(val_mae).round(1).sort_values().to_string())
winner = min(val_mae, key=val_mae.get)
print(f"\nselected on validation: {winner} — ONLY this one may now see the test set")

In [ ]:
final = float(np.abs(candidates[winner].predict(Xte)-yte).mean())
persist = float(np.abs(Xte["lag24"]-yte).mean())
print(f"reported test MAE ({winner}): {final:.1f} MW   [persistence: {persist:.1f} MW]")

## 3. The leakage gallery — how good numbers lie

The most common ways an evaluation flatters itself, all seen in real student and industry work:

1. **Random split of a time series** — tomorrow's neighbours land in the training set; the model "predicts" what it has effectively seen.
2. **Scaling fitted on all data** — the test set's statistics leak into preprocessing (the pipeline above prevents exactly this).
3. **Feature built with future information** — a rolling mean centred on t uses t+1; a daily mean feature computed over the whole day "predicts" 03:00 using 23:00.
4. **Test-set shopping** — trying weeks until the number looks good (the Lab 7 rule returns).
5. **Target leakage** — a feature that is the answer in disguise (total_mw as a feature for ZON_MITT).

Every one of these produces a *better-looking* number. That is precisely why they must be hunted deliberately — nothing in the code smells wrong.

## 4. Choosing the metric is choosing what matters

MAE treats all MW equally; RMSE punishes large misses harder (peak errors matter more to an operator); MAPE breaks near zero and flatters high-load hours; pinball scores quantiles. Your project must *choose and justify* — "we used MAE" is a decision, and it belongs in the decision log.

## Self-check

In [ ]:
assert max(mae_raw, mae_scaled) < 3 * min(mae_raw, mae_scaled), "one MLP variant collapsed unexpectedly"
assert final < persist, "the selected model must beat persistence on the untouched test set"
print("ALL OK — models done, discipline installed. Module 4 hands you an AI assistant; "
      "everything in this notebook is what keeps you the engineer in the room.")